# LSTM over consecutive typing attempts

Each row is one typing attempt with 46 timing features; `repetition` orders attempts within a participant and session. The CSV contains no individual keystroke event/timestamp records, so this notebook models sequences across attempts, not keystrokes within an attempt.

Development uses Session 1 only. The corrected Session 1 data has 99 participants, repetition IDs generally spanning 1–100, and contiguous runs as short as 7 attempts. A window length of 10 is chosen because it yields valid windows for every participant in both chronological partitions while being short enough to retain the shorter contiguous runs where possible. Fit attempts use repetitions ≤70; repetitions 71–79 are embargoed; validation attempts use repetitions ≥80. The nine excluded repetition IDs meet the `window_length - 1` embargo requirement. Partitioning occurs before overlapping windows are formed, and windows never cross participant, partition, or missing-repetition boundaries.

The scaler is fitted only on fit-partition attempt rows. Validation labels use an encoder fitted only on fit-partition participants. The comparison with prior MLP / one-timestep LSTM metrics is descriptive, not a fair head-to-head: the prediction unit and split protocol differ.

In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder, StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA_PATH = PROJECT_DIR / 'data' / 'raw' / 'fixed-text.csv'
PRIOR_VALIDATION_PATH = PROJECT_DIR / 'results' / 'metrics' / 'lstm_one_timestep_validation.csv'
MODEL_OUTPUT_PATH = PROJECT_DIR / 'models' / 'deepkey_lstm_repetition_w10.keras'
METRICS_OUTPUT_PATH = PROJECT_DIR / 'results' / 'metrics' / 'lstm_repetition_w10_validation.csv'

WINDOW_LENGTH = 10
FIT_MAX_REPETITION = 70
VALIDATION_MIN_REPETITION = 80
LSTM_UNITS = 64
DROPOUT_RATE = 0.20
LEARNING_RATE = 0.001
BATCH_SIZE = 256
MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10

if MODEL_OUTPUT_PATH.exists() or METRICS_OUTPUT_PATH.exists():
    raise FileExistsError('Refusing to overwrite an existing sequence-LSTM artifact or metrics file.')

print('TensorFlow:', tf.__version__)
print('Project directory:', PROJECT_DIR)
print('Window length:', WINDOW_LENGTH, '| seed:', SEED)

In [ ]:
df_raw = pd.read_csv(RAW_DATA_PATH)
raw_row_ids = df_raw.index.to_numpy(copy=True)
exclusions = [('p020', 2, 4), ('p071', 1, 20), ('p087', 1, 8)]
df_model = df_raw.copy()
for participant, session, repetition in exclusions:
    excluded = (
        df_model['participant'].eq(participant)
        & df_model['session'].eq(session)
        & df_model['repetition'].eq(repetition)
    )
    if int(excluded.sum()) != 1:
        raise ValueError(f'Expected exactly one documented exclusion: {(participant, session, repetition)}')
    df_model = df_model.loc[~excluded].copy()

excluded_columns = {'participant', 'session', 'repetition', 'total time'}
feature_columns = [column for column in df_model.columns if column not in excluded_columns]
if len(df_model) != 19769 or len(feature_columns) != 46:
    raise ValueError(f'Unexpected corrected modeling data: {df_model.shape}; features={len(feature_columns)}')
if feature_columns != [column for column in df_raw.columns if column not in excluded_columns]:
    raise ValueError('Feature ordering differs from the raw dataset order.')

df_session_1 = df_model.loc[df_model['session'].eq(1)].copy()
df_session_1['_raw_row_id'] = df_session_1.index
if len(df_session_1) != 9876 or df_session_1['participant'].nunique() != 99:
    raise ValueError('Unexpected Session 1 row or participant count.')
if df_session_1.duplicated(['participant', 'session', 'repetition']).any():
    raise ValueError('Duplicate participant/session/repetition keys found.')

run_lengths = []
missing_slots = 0
for participant, group in df_session_1.groupby('participant', sort=True):
    repetitions = sorted(group['repetition'].astype(int).tolist())
    missing_slots += len(set(range(1, 101)) - set(repetitions))
    run_length = 0
    previous = None
    for repetition in repetitions:
        if previous is not None and repetition != previous + 1:
            run_lengths.append(run_length)
            run_length = 0
        run_length += 1
        previous = repetition
    if run_length:
        run_lengths.append(run_length)

print('Corrected rows:', len(df_model), '| Session 1 attempts:', len(df_session_1))
print('Participants:', df_session_1['participant'].nunique(), '| timing features:', len(feature_columns))
print('Contiguous attempt-run lengths: min/median/max =', min(run_lengths), int(np.median(run_lengths)), max(run_lengths))
print('Missing participant-repetition slots among IDs 1..100:', missing_slots)
print('Sequence analysis is restricted to Session 1; Session 2 is not read for evaluation or selection.')

In [ ]:
if VALIDATION_MIN_REPETITION - FIT_MAX_REPETITION - 1 < WINDOW_LENGTH - 1:
    raise ValueError('Chronological boundary embargo is shorter than window_length - 1.')

fit_rows = df_session_1.loc[df_session_1['repetition'].le(FIT_MAX_REPETITION)].copy()
validation_rows = df_session_1.loc[df_session_1['repetition'].ge(VALIDATION_MIN_REPETITION)].copy()
embargo_rows = df_session_1.loc[
    df_session_1['repetition'].between(FIT_MAX_REPETITION + 1, VALIDATION_MIN_REPETITION - 1)
].copy()

fit_participants = set(fit_rows['participant'].unique())
validation_participants = set(validation_rows['participant'].unique())
if fit_participants != validation_participants or len(fit_participants) != 99:
    raise ValueError('Every participant must have attempts in both chronological partitions.')

X_fit_rows = fit_rows[feature_columns].apply(pd.to_numeric, errors='raise').to_numpy(dtype=np.float64)
X_validation_rows = validation_rows[feature_columns].apply(pd.to_numeric, errors='raise').to_numpy(dtype=np.float64)
if not np.isfinite(X_fit_rows).all() or not np.isfinite(X_validation_rows).all():
    raise ValueError('Non-finite feature values found.')

scaler = StandardScaler()
X_fit_scaled = scaler.fit_transform(X_fit_rows).astype(np.float32)
X_validation_scaled = scaler.transform(X_validation_rows).astype(np.float32)
label_encoder = LabelEncoder().fit(fit_rows['participant'])
if set(validation_rows['participant']) - set(label_encoder.classes_):
    raise ValueError('Validation participants include a class absent from the fit partition.')

fit_scaled_by_row = dict(zip(fit_rows['_raw_row_id'].tolist(), X_fit_scaled))
validation_scaled_by_row = dict(zip(validation_rows['_raw_row_id'].tolist(), X_validation_scaled))

def build_windows(partition_rows, scaled_by_row):
    sequences, labels, window_row_ids = [], [], []
    ordered = partition_rows.sort_values(['participant', 'repetition'])
    for participant, group in ordered.groupby('participant', sort=True):
        ids = group['_raw_row_id'].tolist()
        repetitions = group['repetition'].astype(int).tolist()
        run_start = 0
        for position in range(1, len(group) + 1):
            at_run_end = position == len(group) or repetitions[position] != repetitions[position - 1] + 1
            if not at_run_end:
                continue
            run_ids = ids[run_start:position]
            for start in range(0, len(run_ids) - WINDOW_LENGTH + 1):
                row_ids = run_ids[start:start + WINDOW_LENGTH]
                sequences.append(np.stack([scaled_by_row[row_id] for row_id in row_ids]))
                labels.append(label_encoder.transform([participant])[0])
                window_row_ids.append(tuple(row_ids))
            run_start = position
    return np.asarray(sequences, dtype=np.float32), np.asarray(labels, dtype=np.int64), window_row_ids

X_fit, y_fit, fit_window_row_ids = build_windows(fit_rows, fit_scaled_by_row)
X_validation, y_validation, validation_window_row_ids = build_windows(validation_rows, validation_scaled_by_row)
fit_raw_ids = {row_id for window in fit_window_row_ids for row_id in window}
validation_raw_ids = {row_id for window in validation_window_row_ids for row_id in window}
if fit_raw_ids.intersection(validation_raw_ids):
    raise AssertionError('A raw attempt appears in windows in both partitions.')
if not len(X_fit) or not len(X_validation):
    raise ValueError('No valid windows were constructed in one or both partitions.')
if np.unique(y_fit).size != 99 or np.unique(y_validation).size != 99:
    raise ValueError('Both window partitions must contain all 99 encoded participant classes.')
for frame, windows in ((fit_rows, fit_window_row_ids), (validation_rows, validation_window_row_ids)):
    repetition_by_id = frame.set_index('_raw_row_id')['repetition'].astype(int).to_dict()
    participant_by_id = frame.set_index('_raw_row_id')['participant'].to_dict()
    for row_ids in windows:
        if len({participant_by_id[row_id] for row_id in row_ids}) != 1:
            raise AssertionError('Window crosses participant boundaries.')
        reps = [repetition_by_id[row_id] for row_id in row_ids]
        if any(right != left + 1 for left, right in zip(reps, reps[1:])):
            raise AssertionError('Window crosses a missing repetition.')

print('Attempt rows: fit=', len(fit_rows), '| embargo=', len(embargo_rows), '| validation=', len(validation_rows))
print('Valid windows: fit=', len(X_fit), '| validation=', len(X_validation))
print('Window tensor shapes:', X_fit.shape, X_validation.shape)
print('Encoded labels: 0..', len(label_encoder.classes_) - 1, '| scaler fit rows:', int(scaler.n_samples_seen_))
print('Raw attempt overlap between fit and validation windows:', len(fit_raw_ids.intersection(validation_raw_ids)))

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(WINDOW_LENGTH, len(feature_columns))),
    tf.keras.layers.LSTM(LSTM_UNITS),
    tf.keras.layers.Dropout(DROPOUT_RATE),
    tf.keras.layers.Dense(len(label_encoder.classes_), activation='softmax'),
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True,
)
history = model.fit(
    X_fit, y_fit,
    validation_data=(X_validation, y_validation),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=2,
)

validation_probabilities = model.predict(X_validation, batch_size=BATCH_SIZE, verbose=0)
validation_predictions = validation_probabilities.argmax(axis=1)
precision, recall, macro_f1, _ = precision_recall_fscore_support(
    y_validation, validation_predictions, average='macro', zero_division=0,
)
validation_metrics = {
    'accuracy': accuracy_score(y_validation, validation_predictions),
    'balanced_accuracy': balanced_accuracy_score(y_validation, validation_predictions),
    'macro_precision': precision,
    'macro_recall': recall,
    'macro_f1': macro_f1,
}
print('Best epoch:', int(np.argmin(history.history['val_loss']) + 1))
print('Validation metrics:', {key: round(value, 4) for key, value in validation_metrics.items()})

In [ ]:
if MODEL_OUTPUT_PATH.exists() or METRICS_OUTPUT_PATH.exists():
    raise FileExistsError('Refusing to overwrite an existing sequence-LSTM artifact or metrics file.')

prior_validation = pd.read_csv(PRIOR_VALIDATION_PATH)
comparison_names = ['original_mlp_baseline', 'higher_alpha_0.01', 'lstm_one_timestep']
prior_comparison = prior_validation.loc[prior_validation['model'].isin(comparison_names)].copy()
if set(prior_comparison['model']) != set(comparison_names):
    raise ValueError('Prior Session 1 validation comparison rows are incomplete.')
comparison_metrics = [
    'validation_accuracy', 'validation_balanced_accuracy', 'validation_macro_precision',
    'validation_macro_recall', 'validation_macro_f1',
]
comparison = prior_comparison[['model', *comparison_metrics]].copy()
current_row = {'model': 'lstm_repetition_window_10'}
for key, column in zip(
    ['accuracy', 'balanced_accuracy', 'macro_precision', 'macro_recall', 'macro_f1'],
    comparison_metrics,
):
    current_row[column] = validation_metrics[key]
comparison = pd.concat([comparison, pd.DataFrame([current_row])], ignore_index=True)
comparison['prediction_unit_and_split'] = [
    'single attempt; stratified Session 1 row holdout' if model_name != 'lstm_repetition_window_10'
    else '10 consecutive attempts; per-participant chronological holdout with 9-repetition embargo'
    for model_name in comparison['model']
]
display(comparison)

metrics_output = pd.DataFrame([{
    'model': 'lstm_repetition_window_10',
    'window_length': WINDOW_LENGTH,
    'prediction_unit': 'participant identity from 10 consecutive typing attempts',
    'fit_max_repetition': FIT_MAX_REPETITION,
    'validation_min_repetition': VALIDATION_MIN_REPETITION,
    'boundary_embargo_repetitions': VALIDATION_MIN_REPETITION - FIT_MAX_REPETITION - 1,
    'seed': SEED,
    'fit_attempt_rows': len(fit_rows),
    'validation_attempt_rows': len(validation_rows),
    'fit_windows': len(X_fit),
    'validation_windows': len(X_validation),
    'scaler_fit_rows': int(scaler.n_samples_seen_),
    'lstm_units': LSTM_UNITS,
    'dropout_rate': DROPOUT_RATE,
    'learning_rate': LEARNING_RATE,
    'batch_size': BATCH_SIZE,
    'max_epochs': MAX_EPOCHS,
    'best_epoch': int(np.argmin(history.history['val_loss']) + 1),
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    **{f'validation_{key}': value for key, value in validation_metrics.items()},
}])
MODEL_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
model.save(MODEL_OUTPUT_PATH)
metrics_output.to_csv(METRICS_OUTPUT_PATH, index=False)
print('Saved model:', MODEL_OUTPUT_PATH)
print('Saved validation metrics:', METRICS_OUTPUT_PATH)
print('Session 2 was not used or evaluated.')